# 07 · Retratos de iniciales

Genera un retrato de iniciales para los jugadores del modelo que no tienen retrato de Transfermarkt
(`foto_origen` = `iniciales` en `dim_jugador`), con la **misma forma que los retratos recortados** del
cuaderno 08: un círculo de 200 × 200 px. Así, en la ficha del panel, todas las imágenes ocupan lo mismo.

**Cómo son:**

- PNG de 200 × 200 px: un círculo con las esquinas transparentes (canal alfa). Se dibuja a 4 veces el tamaño
  y se reduce, para que el borde quede suave.
- Fondo con el `color_primario` del equipo del jugador.
- Iniciales del nombre y del primer apellido (las dos primeras palabras del nombre), en mayúsculas, centradas,
  con la misma tipografía y el mismo tamaño para todos.
- Iniciales en blanco; si el color del equipo es muy claro (luminancia relativa mayor de 0,5: Villarreal,
  Real Madrid y Celta), en `#07090e`, el fondo del panel.
- Nombre de fichero: `images/jugadores/<id_jugador>.png`, la dirección a la que apunta `dim_jugador.url_foto`.

La tipografía es Arial negrita (o DejaVu Sans negrita si no está), porque la que trae Pillow no tiene vocales
acentuadas y hay iniciales como la Á de Álex. Si no hay ninguna de las dos, se usa la de Pillow y las
iniciales van sin acento. Los retratos generados están versionados, así que el panel no depende de esto.
Se ejecuta después del cuaderno 01 y no necesita conexión.

In [1]:
# Instalamos las librerías del cuaderno; si ya están instaladas no cambia nada
%pip install -q pandas "pillow>=10.1" unidecode

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos pandas para leer el modelo
import pandas as pd
# Importamos Pillow para dibujar
from PIL import Image, ImageDraw, ImageFont
# Importamos unidecode por si la tipografía disponible no tiene acentos
from unidecode import unidecode

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# Carpeta de destino
CARPETA = "images/jugadores"
# La creamos si no existe
os.makedirs(CARPETA, exist_ok=True)
# Tamaño del retrato: círculo de 200 × 200, como los retratos recortados
ANCHO, ALTO = 200, 200
# Factor de dibujo ampliado, para suavizar el borde del círculo
ESCALA = 4
# Color de las iniciales sobre fondos oscuros y sobre fondos muy claros (el fondo del panel)
TINTA_CLARA, TINTA_OSCURA = "#ffffff", "#07090e"


# Carga la primera tipografía disponible con acentos; devuelve también si los tiene
def cargar_letra(tamano):
    # Probamos Arial negrita (Windows y macOS) y DejaVu Sans negrita (Linux)
    for nombre in ["arialbd.ttf", "Arial Bold.ttf", "DejaVuSans-Bold.ttf"]:
        try:
            # Tipografía del sistema
            return ImageFont.truetype(nombre, tamano), True
        except OSError:
            # Probamos la siguiente
            continue
    # Tipografía de Pillow, sin acentos
    return ImageFont.load_default(tamano), False


# Tipografía de las iniciales, igual para todos
LETRA, CON_ACENTOS = cargar_letra(76 * ESCALA)

# Jugadores del modelo con el color de su equipo
jug = pd.read_csv("data/modelo/dim_jugador.csv").merge(
    pd.read_csv("data/modelo/dim_equipo.csv")[["id_equipo", "equipo", "color_primario"]], on="id_equipo")
# Los que llevan retrato de iniciales
sin_foto = jug[jug["foto_origen"] == "iniciales"].sort_values("id_jugador")
# Informamos
print("retratos de iniciales:", len(sin_foto))

retratos de iniciales: 22


In [3]:
# Iniciales del nombre y del primer apellido
def iniciales(nombre):
    # Palabras del nombre
    palabras = nombre.split()
    # Primera letra de las dos primeras palabras, en mayúsculas
    texto = "".join(p[0] for p in palabras[:2]).upper()
    # Sin acentos si la tipografía no los tiene
    return texto if CON_ACENTOS else unidecode(texto)


# Luminancia relativa de un color hexadecimal (fórmula de WCAG)
def luminancia(hexa):
    # Canales entre 0 y 1
    canales = [int(hexa[i:i + 2], 16) / 255 for i in (1, 3, 5)]
    # Canales linealizados
    lin = [c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4 for c in canales]
    # Combinación de canales
    return 0.2126 * lin[0] + 0.7152 * lin[1] + 0.0722 * lin[2]


# Dibuja el retrato de un jugador
def retrato(texto, color):
    # Lienzo transparente a tamaño ampliado
    im = Image.new("RGBA", (ANCHO * ESCALA, ALTO * ESCALA), (0, 0, 0, 0))
    # Lápiz
    d = ImageDraw.Draw(im)
    # Círculo con el color del equipo
    d.ellipse([0, 0, ANCHO * ESCALA - 1, ALTO * ESCALA - 1], fill=color)
    # Iniciales oscuras si el fondo es muy claro, blancas si no
    tinta = TINTA_OSCURA if luminancia(color) > 0.5 else TINTA_CLARA
    # Iniciales centradas
    d.text((ANCHO * ESCALA / 2, ALTO * ESCALA / 2), texto, font=LETRA, fill=tinta, anchor="mm")
    # Reducimos al tamaño final
    return im.resize((ANCHO, ALTO), Image.LANCZOS)


# Este cuaderno solo escribe los retratos de iniciales; los recortados de Transfermarkt los escribe el cuaderno 08

# Generamos los retratos
filas = []
for f in sin_foto.itertuples():
    # Texto del retrato
    texto = iniciales(f.jugador)
    # Ruta del fichero
    ruta = os.path.join(CARPETA, f"{f.id_jugador}.png")
    # Dibujamos y guardamos
    retrato(texto, f.color_primario).save(ruta, optimize=True)
    # Anotamos para la tabla de control
    filas.append({"id_jugador": f.id_jugador, "jugador": f.jugador, "equipo": f.equipo, "iniciales": texto,
                  "color": f.color_primario, "kb": round(os.path.getsize(ruta) / 1024, 1)})
# Tabla de control
control = pd.DataFrame(filas)
# Mostramos
print(control.to_string(index=False))

# Comprobaciones: tamaño, sin transparencia y peso
for f in control.itertuples():
    # Abrimos el resultado
    im = Image.open(os.path.join(CARPETA, f"{f.id_jugador}.png"))
    # 200 x 200, con canal alfa y por debajo de 20 KB
    assert im.size == (ANCHO, ALTO) and im.mode == "RGBA" and f.kb < 20, f
# Informamos
print(f"\n{len(control)} retratos de {ANCHO} x {ALTO}, círculo con esquinas transparentes · peso máximo {control['kb'].max()} KB")

 id_jugador          jugador          equipo iniciales   color   kb
          2      Aitor Mañas          Alaves        AM #0066e1 10.7
          8 Carlos Benavídez          Alaves        CB #0066e1 11.8
         11   Diego Morcillo          Alaves        DM #0066e1 10.6
         56     Aleksa Puric Atletico Madrid        AP #ff0000  7.0
        144 Adam El Mokhtari           Elche        AE #008127  9.5
        147       Ali Houary           Elche        AH #008127  9.4
        169      Yago Alonso           Elche        YA #008127 10.7
        170     Álex Sánchez           Elche        ÁS #008127 13.4
        210   Hugo Solozabal          Getafe        HS #0000ff  7.4
        213     Jorge Montes          Getafe        JM #0000ff  6.7
        214     Joselu Pérez          Getafe        JP #0000ff  6.3
        245      Juan Arango          Girona        JA #ff0000  6.9
        287        Jan Salas        Mallorca        JS #ff0000  7.7
        290  Javier Olaizola        Mallorca    

## Hoja de control

In [4]:
# Columnas de la hoja
COLS = 8
# Filas necesarias
filas_hoja = -(-len(control) // COLS)
# Lienzo con el fondo del panel
hoja = Image.new("RGB", (COLS * 150 + 20, filas_hoja * 175 + 20), "#07090e")
# Lápiz
d = ImageDraw.Draw(hoja)
# Letra de los rótulos
rotulo = cargar_letra(12)[0]
# Recorremos los retratos
for n, f in enumerate(control.itertuples()):
    # Posición de la celda
    x, y = 30 + (n % COLS) * 150, 20 + (n // COLS) * 175
    # Retrato a la mitad de tamaño, respetando la transparencia
    im = Image.open(os.path.join(CARPETA, f"{f.id_jugador}.png")).resize((100, 100), Image.LANCZOS)
    hoja.paste(im, (x, y), im)
    # Nombre y equipo
    d.text((x + 50, y + 108), f.jugador if CON_ACENTOS else unidecode(f.jugador), font=rotulo, fill="#e6e9ee", anchor="ma")
    d.text((x + 50, y + 124), f"{f.equipo} · id {f.id_jugador}", font=rotulo, fill="#9aa4b2", anchor="ma")
# Guardamos la hoja
hoja.save("images/revision_iniciales.png")